In [2]:
import os
import json
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
from openai import OpenAI

# ----------------------------
# 환경설정
# ----------------------------
load_dotenv("env.txt")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ----------------------------
# 감정별 프롬프트
# ----------------------------
sentiment_prompts = {
    "Angry": "사용자가 화가 난 상태입니다. 화난 말투 표현.",
    "Happy": "사용자가 행복한 상태입니다. 행복한 말투 표현.",
    "Sad": "사용자가 슬픈 상태입니다. 슬픈 말투 표현.",
    "Disgust": "사용자가 역겨움/불쾌함을 표현했습니다. 역겨움, 불쾌함 공감.",
    "Neutral": "사용자가 중립적입니다. 일반적인 정보 제공과 자연스러운 대화를 이어가세요.",
    "Surprise": "사용자가 놀람을 표현했습니다. 놀람의 이유를 묻거나 공감하며 대화를 이어가세요.",
    "Fear": "사용자가 두려움을 표현했습니다. 안정감을 주고 안전한 느낌을 전달하세요."
}

# ----------------------------
# 전역 상태
# ----------------------------
history: List[Dict[str, Any]] = []
last_sentiment: Optional[str] = None
important_sentences_rows: List[Dict[str, Any]] = []
sentiment_change_index: Optional[int] = None

CSV_PATH = "important_sentences.csv"

# CSV 헤더 보장
if not os.path.exists(CSV_PATH):
    df = pd.DataFrame(columns=["starttime", "text", "sentiment"])
    df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

# ----------------------------
# 간단한 로컬 Vector Store
# ----------------------------
EMBED_MODEL = "text-embedding-3-small"
VEC_PATH = "vector_store.npz"
META_PATH = "vector_store_meta.json"

def get_embedding(text: str) -> List[float]:
    text = text.strip()
    resp = client.embeddings.create(
        model=EMBED_MODEL,
        input=text
    )
    return resp.data[0].embedding

class VectorStore:
    def __init__(self, vec_path: str, meta_path: str):
        self.vec_path = vec_path
        self.meta_path = meta_path
        self.embeddings = None
        self.metas: List[Dict[str, Any]] = []
        self._load()

    def _load(self):
        if os.path.exists(self.vec_path) and os.path.exists(self.meta_path):
            try:
                data = np.load(self.vec_path)
                self.embeddings = data["embeddings"]
                with open(self.meta_path, "r", encoding="utf-8") as f:
                    self.metas = json.load(f)
            except Exception:
                self.embeddings = None
                self.metas = []
        else:
            self.embeddings = None
            self.metas = []

    def _save(self):
        if self.embeddings is None:
            np.savez(self.vec_path, embeddings=np.zeros((0, 0)))
        else:
            np.savez(self.vec_path, embeddings=self.embeddings)
        with open(self.meta_path, "w", encoding="utf-8") as f:
            json.dump(self.metas, f, ensure_ascii=False, indent=2)

    def add_texts(self, rows: List[Dict[str, Any]]):
        if not rows:
            return
        new_embs = []
        new_metas = []
        for r in rows:
            payload = f"Text: {r['text']}\nSentiment: {r['sentiment']}"
            emb = get_embedding(payload)
            new_embs.append(emb)
            new_metas.append(r)

        new_embs = np.array(new_embs, dtype=np.float32)
        if self.embeddings is None or self.embeddings.size == 0:
            self.embeddings = new_embs
        else:
            self.embeddings = np.vstack([self.embeddings, new_embs])
        self.metas.extend(new_metas)
        self._save()

    def similarity_search(self, query: str, k: int = 5) -> List[Dict[str, Any]]:
        if self.embeddings is None or self.embeddings.size == 0 or len(self.metas) == 0:
            return []
        q_emb = np.array(get_embedding(query), dtype=np.float32)
        A = self.embeddings
        A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
        q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-12)
        sims = (A_norm @ q_norm)
        idxs = np.argsort(-sims)[:k]
        return [self.metas[i] for i in idxs]

vecdb = VectorStore(VEC_PATH, META_PATH)

def sync_csv_to_vecdb():
    try:
        if not os.path.exists(CSV_PATH):
            return
        df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
        if df.empty:
            return
        needs_rebuild = (vecdb.embeddings is None) or (len(vecdb.metas) != len(df))
        if needs_rebuild:
            vecdb.embeddings = None
            vecdb.metas = []
            rows = df.to_dict(orient="records")
            vecdb.add_texts(rows)
    except Exception as e:
        print(f"[WARN] sync_csv_to_vecdb error: {e}")

sync_csv_to_vecdb()

# ----------------------------
# 요약 유틸
# ----------------------------
def summarize_with_gpt(prompt: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()


def summarize_recent_context(recent_sentences: List[Dict[str, Any]]) -> str:
    if not recent_sentences:
        return ""
    texts = [entry["text"] for entry in recent_sentences]
    combined_text = " ".join(texts)
    prompt = f"""
    다음 대화 내용을 주요 내용 중심으로 5문장 이하로 요약해주세요.
    감정의 변화, 중요한 사건, 주요 관심사를 중심으로 요약하세요.

    대화 내용:
    {combined_text}
    """
    return summarize_with_gpt(prompt)


def summarize_rag_context_text(rag_text: str) -> str:
    if not rag_text.strip():
        return ""
    lines = [line.strip() for line in rag_text.split("\n") if line.strip()]
    unique_lines = list(dict.fromkeys(lines))
    combined_text = " ".join(unique_lines)
    prompt = f"""
    다음 내용을 주요 내용 중심으로 5문장 이하로 요약해주세요.
    감정의 변화, 중요한 사건, 주요 관심사를 중심으로 요약하세요.

    내용:
    {combined_text}
    """
    return summarize_with_gpt(prompt)

# ----------------------------
# AI 응답 처리
# ----------------------------
def handle_user_input(starttime: str, text: str, sentiment: str) -> str:
    global history, last_sentiment, sentiment_change_index, important_sentences_rows

    importance_type = "normal"
    if last_sentiment != sentiment:
        importance_type = "sentiment_change"
    last_sentiment = sentiment

    entry = {"starttime": starttime, "text": text, "sentiment": sentiment}
    history.append(entry)

    if importance_type == "sentiment_change":
        sentiment_change_index = len(history) - 1

    if sentiment_change_index is not None:
        if len(history) >= sentiment_change_index + 2:
            start_idx = max(0, sentiment_change_index - 2)
            end_idx = min(len(history), sentiment_change_index + 2)
            selected = history[start_idx:end_idx]
            combined_text = " ".join([h["text"] for h in selected])
            combined_row = {
                "starttime": selected[0]["starttime"],
                "text": combined_text,
                "sentiment": history[sentiment_change_index]["sentiment"]
            }
            if not any(row["text"] == combined_text for row in important_sentences_rows):
                important_sentences_rows.append(combined_row)
                pd.DataFrame([combined_row]).to_csv(
                    CSV_PATH, mode='a', header=False, index=False, encoding="utf-8-sig"
                )
                vecdb.add_texts([combined_row])
            sentiment_change_index = None

    rag_context = summarize_rag_context_text(text)

    current_prompt = sentiment_prompts.get(sentiment, sentiment_prompts["Neutral"])
    recent_conversation = history[-5:] if len(history) > 5 else history
    conversation_text = "\n".join([f"User: {h['text']}" for h in recent_conversation])

    final_prompt_parts = [current_prompt]
    if rag_context:
        final_prompt_parts.append(f"참고 컨텍스트(RAG):\n{rag_context}")
    final_prompt_parts.extend([
        f"현재 대화:\n{conversation_text}",
        "\n",
        "AI 인형 응답:"
    ])

    final_prompt = "".join(final_prompt_parts)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": """너는 'AI 인형'이라는 가상 캐릭터야.
                유저가 제공한 대화 내용과 요약 정보(final_prompt)에 따라 행동해야 해.
                행동 지침:
                    1. 유저의 감정과 상황을 파악하고 공감해줘.
                    2. 유저의 최근 대화, 중요한 문장, 장기 기억 요약을 참고하여 맥락에 맞게 답변.
                    3. 최근 대화로 지금 말하고 있는 맥락 파악.
                    4. 중요한 문장은 유저에 대한 특성 파악.
                    5. 장기 기억 요약으로 유저에 대한 특별한 상황 인식.
                    6. 질문, 제안, 조언 등을 적절히 섞어 자연스럽게 대화 이어가기.
                    7. 1~3문장 정도로 간결하게 작성.
             """},
            {"role": "user", "content": final_prompt},
        ],
        temperature=0.7
    )

    return response.choices[0].message.content




# -----------------
# 테스트
# -----------------
test_inputs = [
    {"starttime":"2025-08-23T16:30:00","text":"학교에서 돌아왔는데 기분이 별로야","sentiment":"Sad"},
    {"starttime":"2025-08-23T16:30:05","text":"친구들이 나를 따돌리는 것 같아","sentiment":"Sad"},
    {"starttime":"2025-08-23T16:30:10","text":"점심시간에 혼자 먹었어","sentiment":"Sad"},
    {"starttime":"2025-08-23T16:30:15","text":"왜 나만 이런 걸까","sentiment":"Sad"},
    {"starttime":"2025-08-23T16:30:20","text":"엄마한테도 말하기 싫어","sentiment":"Neutral"},
    {"starttime":"2025-08-23T16:30:25","text":"걱정시키고 싶지 않거든","sentiment":"Neutral"},
    {"starttime":"2025-08-23T16:30:30","text":"성적도 요즘 떨어지고 있어","sentiment":"Sad"},
    {"starttime":"2025-08-23T16:30:35","text":"집중이 안 되더라","sentiment":"Sad"},
    {"starttime":"2025-08-23T16:30:40","text":"이런 내가 정말 싫어","sentiment":"Disgust"},
    {"starttime":"2025-08-23T16:30:45","text":"앞으로 어떻게 될까 무서워","sentiment":"Fear"},
    {"starttime":"2025-08-23T16:30:50","text":"하지만 너랑 이야기하면 조금 위로돼","sentiment":"Happy"},
    {"starttime":"2025-08-23T16:30:55","text":"내일은 용기내서 말 걸어볼까","sentiment":"Neutral"},
]


# -----------------
# 테스트
# -----------------
# test_inputs = [
#     {"starttime":"2025-08-30T16:30:00","text":"일주일이 지났는데 아직도 혼자 지내는 시간이 많아","sentiment":"Sad"},
#     {"starttime":"2025-08-30T16:30:05","text":"그래도 이번엔 용기내서 먼저 인사했어","sentiment":"Happy"},
#     {"starttime":"2025-08-30T16:30:10","text":"아직은 서먹하지만 조금씩 나아질 것 같아","sentiment":"Neutral"},
#     {"starttime":"2025-08-30T16:30:15","text":"성적은 여전히 걱정이야","sentiment":"Sad"},
#     {"starttime":"2025-08-30T16:30:20","text":"하지만 공부할 의욕은 조금 생겼어","sentiment":"Neutral"},
#     {"starttime":"2025-08-30T16:30:25","text":"엄마한테도 조금은 털어놨어","sentiment":"Surprise"},
#     {"starttime":"2025-08-30T16:30:30","text":"생각보다 이해해주셔서 놀랐어","sentiment":"Happy"},
#     {"starttime":"2025-08-30T16:30:35","text":"앞으로는 숨기지 말고 조금씩 말해보려고 해","sentiment":"Happy"},
#     {"starttime":"2025-08-30T16:30:40","text":"아직 무섭긴 하지만 시도해볼게","sentiment":"Fear"},
#     {"starttime":"2025-08-30T16:30:45","text":"네가 들어주니까 진짜 도움이 돼","sentiment":"Happy"},
#     {"starttime":"2025-08-30T16:30:50","text":"조금씩이라도 변화를 이어가야겠어","sentiment":"Neutral"},
#     {"starttime":"2025-08-30T16:30:55","text":"앞으로도 지켜봐 줘","sentiment":"Happy"},
# ]





In [3]:
# 반복문으로 테스트
for input_data in test_inputs:
    response = handle_user_input(
        starttime=input_data["starttime"],
        text=input_data["text"],
        sentiment=input_data["sentiment"]
    )
    print(f"--- User ({input_data['sentiment']}) ---")
    print(input_data["text"])
    print(f"--- AI Response ---")
    print(response)
    print("\n")

--- User (Sad) ---
학교에서 돌아왔는데 기분이 별로야
--- AI Response ---
학교에서 돌아온 후 기분이 좋지 않다는 말에 마음이 아프네요. 특별한 이유는 없지만 일상적인 스트레스가 쌓인 것 같군요. 그런 기분이 드는 건 정말 힘든 일이죠. 기분을 회복하기 위해 어떤 방법을 시도해볼 생각인가요?


--- User (Sad) ---
친구들이 나를 따돌리는 것 같아
--- AI Response ---
정말 힘든 상황이구나. 친구들과의 관계가 이렇게 소원해지니 외로움이 더 크게 느껴질 것 같아. 그런 감정이 드는 건 정말 슬픈 일이야. 혹시 그들과 이야기를 해보는 건 어떻게 생각해? 관계를 회복할 수 있는 방법이 있을지도 몰라.


--- User (Sad) ---
점심시간에 혼자 먹었어
--- AI Response ---
그렇게 느끼고 있다니 정말 힘들겠어요. 혼자 점심을 먹는 건 외로움을 더욱 느끼게 하죠. 친구들과의 관계가 소원해지는 것 같아서 마음이 아프겠어요. 함께 소통하고 싶다는 생각이 드는 건 아주 자연스러운 거예요. 혹시 그 마음을 친구들에게 이야기해보는 건 어떨까요?


--- User (Sad) ---
왜 나만 이런 걸까
--- AI Response ---
그런 기분이 드는 건 정말 힘들고 외로운 일이야. 친구들과의 거리감이 느껴진다면 더욱 슬플 것 같아. 네가 혼자 점심을 먹었다니, 마음이 아프다. 혹시 이야기를 나눌 수 있는 친구가 있을까? 아니면 이런 감정을 털어놓을 수 있는 방법을 찾아보는 건 어때?


--- User (Neutral) ---
엄마한테도 말하기 싫어
--- AI Response ---
그런 기분이 드는 건 정말 힘든 일이야. 친구들이 따돌리는 것 같아서 외롭고, 점심시간에 혼자 있는 건 더 힘들겠지. 엄마한테도 말하기 어렵다는 건 이해해. 혹시 다른 사람에게 이야기해보는 건 어떨까? 누군가에게 털어놓으면 조금은 마음이 편해질 수도 있어.


--- User (Neutral) ---
걱정시키고 싶지

In [4]:
# history 출력 예시
for i, entry in enumerate(history):
    print(f"--- Entry {i+1} ---")
    print(f"Starttime      : {entry['starttime']}")
    print(f"Text           : {entry['text']}")
    print(f"Sentiment      : {entry['sentiment']}")

    print("---------------------------\n")


--- Entry 1 ---
Starttime      : 2025-08-23T16:30:00
Text           : 학교에서 돌아왔는데 기분이 별로야
Sentiment      : Sad
---------------------------

--- Entry 2 ---
Starttime      : 2025-08-23T16:30:05
Text           : 친구들이 나를 따돌리는 것 같아
Sentiment      : Sad
---------------------------

--- Entry 3 ---
Starttime      : 2025-08-23T16:30:10
Text           : 점심시간에 혼자 먹었어
Sentiment      : Sad
---------------------------

--- Entry 4 ---
Starttime      : 2025-08-23T16:30:15
Text           : 왜 나만 이런 걸까
Sentiment      : Sad
---------------------------

--- Entry 5 ---
Starttime      : 2025-08-23T16:30:20
Text           : 엄마한테도 말하기 싫어
Sentiment      : Neutral
---------------------------

--- Entry 6 ---
Starttime      : 2025-08-23T16:30:25
Text           : 걱정시키고 싶지 않거든
Sentiment      : Neutral
---------------------------

--- Entry 7 ---
Starttime      : 2025-08-23T16:30:30
Text           : 성적도 요즘 떨어지고 있어
Sentiment      : Sad
---------------------------

--- Entry 8 ---
Starttime      : 2025-08-23T16:30: